# XGBoost Classification and SHAP Interpretation

An XGBoost classifier was trained to distinguish place from active non-place cells and silent cells using spatial selectivity–associated genes. Model performance was evaluated under a leave-one-mouse-out (LOMO) cross-validation framework. 

To quantify gene-level contributions to classification, SHAP (SHapley Additive exPlanations) values were computed for each test fold, enabling identification of genes driving model predictions.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
from datetime import date
import scanpy as sc
import omicverse as ov
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import sparse
import pickle as pkl

import xgboost as xgb
import shap
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

SAVE = False  # whether to save the outputs of this notebook

In [ ]:
mpl.rcParams["font.family"] = "Arial"          
mpl.rcParams["font.sans-serif"] = ["Arial"]    
mpl.rcParams["pdf.fonttype"] = 42              
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["figure.dpi"]  = 300

---

### 1. Data Preparation

Spatial selectivity–associated genes and the corresponding AnnData object were loaded for downstream analysis.

---

In [ ]:
input_path = r"F:\Code\codex\PlaceCell_snRNAseq\results"
output_path = Path(r"F:\Code\codex\PlaceCell_snRNAseq\results\XGBoost_SHAP")
output_path.mkdir(parents=True, exist_ok=True)
today = date.today().strftime("%Y%m%d")

spatial_selectivity_genes = pd.read_csv(os.path.join(input_path,"spatial_candidate_final_genes_20260609.csv"),index_col= 0)

In [ ]:
spatial_selectivity_genes.head()

In [ ]:
adata = sc.read_h5ad(r"F:\Data\snRNAseq\PlaceCell\FilteredData\PCAnalysis_all_QC_20260127.h5ad")
adata_annotation = sc.read_h5ad(
    r"F:\Code\codex\PlaceCell_snRNAseq\Data\FilteredData\All_QC_filtered_annotation_20260604.h5ad"
)
assert adata.n_obs == adata_annotation.n_obs
assert (adata.obs_names == adata_annotation.obs_names).all()

# 转移 cell type 注释
adata.obs["cell_types"] = adata_annotation.obs["cell_types"].values

# 去掉 Unassigned
adata = adata[adata.obs["cell_types"] == "CA1-ProS"].copy()

In [ ]:
# Normalize in place
sc.pp.normalize_total(adata, target_sum=1e6)   # A higher target_sum enhances visual contrast
# Log-transform
sc.pp.log1p(adata)

In [ ]:
adata.X.max()

In [ ]:
adata_pn = adata[adata.obs["sample"].isin(["PC94", "PC110", "PC112", "PC114"])].copy()   #place vs nonplace
adata_ps = adata[adata.obs["sample"].isin(["PC103", "PC106", "PC118"])].copy() #place vs silent
adata_ns = adata[adata.obs["sample"].isin(["PC92", "PC105", "PC132"])].copy()  #nonplace vs silent

---

### 2. XGboost predication

Spatial selectivity–associated genes and the corresponding AnnData object were loaded for downstream analysis.

---

In [ ]:
genes = spatial_selectivity_genes["gene"].to_list()
len(genes)

In [ ]:
# random select the same number of spatial selectiivty genes
X = adata.X
spatial = pd.Index(genes).dropna().unique()

df = pd.DataFrame({
    "gene": adata.var_names,
    "frac": np.asarray((X > 0).mean(0)).ravel(),
    "mean": np.asarray(X.mean(0)).ravel(),
})
df = df[df["frac"] > 0.05].copy()
df["sp"] = df["gene"].isin(spatial)
df["bin"] = pd.qcut(df["mean"], q=10, duplicates="drop")

need = df[df["sp"]].groupby("bin").size()
random_genes = (
    df[~df["sp"]].groupby("bin", group_keys=False)
      .apply(lambda g: g.sample(n=need.get(g.name, 0), random_state=42))
      ["gene"].tolist()
)


In [ ]:
len(random_genes)

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "scripts" / "ml_xgboost_lomo.py").exists():
            return path
    raise FileNotFoundError("Could not find scripts/ml_xgboost_lomo.py")


PROJECT_ROOT = _find_project_root()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from ml_xgboost_lomo import (
    compute_lomo_shap,
    fit_lomo_xgb,
    run_label_shuffle_null_parallel,
    run_random_gene_null_fast,
)

In [ ]:
res_pn = fit_lomo_xgb(adata_pn, genes, verbose=True)
print("Simple Mean AUC:", res_pn["simple_mean_auc"])
print("Weighted Mean AUC:", res_pn["weighted_mean_auc"])

In [ ]:
res_ps = fit_lomo_xgb(adata_ps, genes, verbose=True)
print("Simple Mean AUC:", res_ps["simple_mean_auc"])
print("Weighted Mean AUC:", res_ps["weighted_mean_auc"])

In [ ]:
res_ns = fit_lomo_xgb(adata_ns, genes, positive_label="nonplace", verbose=True)
print("Simple Mean AUC:", res_ns["simple_mean_auc"])
print("Weighted Mean AUC:", res_ns["weighted_mean_auc"])

In [ ]:
# Random-gene null helper is imported from scripts/ml_xgboost_lomo.py.
# Keeping the implementation in one module avoids duplicated XGBoost parameters across notebook cells.

In [ ]:
rand_aucs = run_random_gene_null_fast(adata_ns, real_genes=genes, n_random=1000, n_jobs=28, experiment="nonplace")

In [ ]:
# Label-shuffle null helper is imported from scripts/ml_xgboost_lomo.py.
# The shared implementation skips invalid folds and uses single-threaded models under outer joblib parallelism.

In [ ]:
shuf_aucs = run_label_shuffle_null_parallel(adata_ns, genes, n_shuffle=1000, n_jobs=28, positive_label="nonplace")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import matplotlib as mpl
import os

# ===== 全局风格 =====
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.linewidth'] = 1
mpl.rcParams['font.size'] = 8


def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


def plot_null_distribution(null_vals, real_auc, title):

    mean_null = np.mean(null_vals)
    std_null  = np.std(null_vals)
    z = (real_auc - mean_null) / std_null
    p_emp = (np.sum(null_vals >= real_auc) + 1) / (len(null_vals) + 1)

    star = p_to_star(p_emp)

    fig, ax = plt.subplots(figsize=(4,2.5), dpi=600)

    # Histogram
    ax.hist(
        null_vals,
        bins=30,
        density=True,
        color="lightgray",
        edgecolor="none",
        alpha=0.9
    )

    # Gaussian fit
    x = np.linspace(min(null_vals), max(null_vals), 600)
    pdf = norm.pdf(x, mean_null, std_null)
    ax.plot(x, pdf, color="black", linewidth=1)

    ax.set_xlim(0.2, 0.8)

    # Real AUC vertical line
    ax.axvline(real_auc, color="#1f77b4", linewidth=2)

    # ⭐ 显著性星号放在 real AUC 上方
    ymax = ax.get_ylim()[1]
    ax.text(
        real_auc,
        ymax * 0.92,
        star,
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

    # # 统计信息（简化版更专业）
    # ax.text(
    #     0.05,
    #     0.95,
    #     f"Real AUC = {real_auc:.3f}\n"
    #     f"Null mean = {mean_null:.3f}\n"
    #     f"Z = {z:.2f}\n"
    #     f"p = {p_emp:.3f}",
    #     transform=ax.transAxes,
    #     verticalalignment="top"
    # )

    # 去掉右上边框
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_xlabel("AUC")
    ax.set_ylabel("Density")
    ax.set_title(title)

    plt.tight_layout()
    plt.savefig(
        os.path.join(output_path, f"Random_distribution_{title}_{today}.svg"),
        bbox_inches="tight"
    )
    plt.show()

In [ ]:
# mpl.rcParams['svg.fonttype'] = 'path'
plot_null_distribution(rand_aucs, res_ns["simple_mean_auc"], "ANP vs S")

In [ ]:
shap_df = compute_lomo_shap(res_ps["X"], res_ps["groups"], res_ps["models"], genes)
print(shap_df.head(50))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from matplotlib.colors import LinearSegmentedColormap
# 先准备数据
X = adata_pn[:, genes].X
if not isinstance(X, np.ndarray):
    X = X.toarray()

y = (adata_pn.obs["condition"] == "place").astype(int).values
groups = adata_pn.obs["sample"].values
unique_mice = np.unique(groups)

all_sv, all_Xtest = [], []

for test_mouse in unique_mice:   # 也可写成 for test_mouse in np.unique(groups):
    train_mask = groups != test_mouse
    test_mask  = groups == test_mouse

    X_train, X_test = X[train_mask], X[test_mask]
    y_train = y[train_mask]

    model = XGBClassifier(
        n_estimators=700,
        learning_rate=0.01,
        max_depth=5,
        min_child_weight=2,   # 显式写出来
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=0.8,
        eval_metric="auc",
        n_jobs=-1,
        # random_state=random_state,
    )

    model.fit(X_train, y_train)

    sv = shap.TreeExplainer(model)(X_test).values
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    all_sv.append(sv)
    all_Xtest.append(X_test)

SV = np.vstack(all_sv)
XT = np.vstack(all_Xtest)
colors = ["#22456F", "#528FAC","#AECAD8","#F2F2F2","#F7D29D","#EEB84B", "#E87A2D"]  # 示例颜色
cmap_custom = LinearSegmentedColormap.from_list("custom_cmap", colors, N=256)
shap.summary_plot(SV, XT, feature_names=genes, show=False, cmap=cmap_custom, max_display=20)
plt.tight_layout()
plt.savefig(os.path.join(output_path, f"SHAP_beamstream_{today}.svg"), dpi = 600)
plt.show()


In [ ]:
spatial_selectivity_genes[spatial_selectivity_genes['label'] == "Down"]

In [ ]:
genes_used = spatial_selectivity_genes["gene"].tolist()
print(len(genes_used))   # 期待 89

In [ ]:
# 在训练/LOMO 之前
genes_used = list(adata_pn[:, genes].var_names)   # 冻结当时用于训练的特征名（顺序很重要）
X = _to_dense(adata_pn[:, genes_used].X)          # 用 genes_used 构建 X（不要再用 genes 变量）

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt

mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.linewidth'] = 1
mpl.rcParams['font.size'] = 16
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8

def plot_shap_dependence_grid_np(
    XT: np.ndarray,
    SV: np.ndarray,
    feature_names,
    genes,
    ncols=2,
    figsize=(12, 12),
    dpi=300,
    point_size=6,
    alpha=0.8,
    color_by=None,      # "self" or gene name or None
    cmap="viridis",
    add_colorbar=True,    # 若 color_by=None 会自动忽略
    point_color="#6FA3C6"
):
    name_to_idx = {g: i for i, g in enumerate(feature_names)}
    n = len(genes)
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=dpi, sharey=False)
    axes = np.array(axes).reshape(-1)
    
    for ax, g in zip(axes, genes):
        if g not in name_to_idx:
            ax.set_title(f"{g} (missing)")
            ax.axis("off")
            continue

        j = name_to_idx[g]
        x = XT[:, j]
        y = SV[:, j]

        # --- decide coloring ---
        if color_by is None:
            sc = ax.scatter(x, y, color=point_color, s=point_size, alpha=alpha, linewidths=0)
        else:
            if color_by == "self":
                c = x
                c_label = "Gene expression"
            else:
                if color_by not in name_to_idx:
                    raise ValueError(f"color_by='{color_by}' not found in feature_names")
                c = XT[:, name_to_idx[color_by]]
                c_label = f"{color_by} expression"

            sc = ax.scatter(x, y, c=c, cmap=cmap, s=point_size, alpha=alpha, linewidths=0)

            if add_colorbar:
                cb = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
                cb.set_label(c_label)

        ax.set_title(g)
        ax.set_xlabel("Gene expression")
        ax.set_ylabel("SHAP value")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for ax in axes[len(genes):]:
        ax.axis("off")

    plt.savefig(
        os.path.join(output_path, f"SHAP_expression_{genes}_{today}.svg"),
        bbox_inches="tight"
    )
        
    plt.tight_layout()

    return fig


In [ ]:
genes_used = spatial_selectivity_genes["gene"].tolist()

feature_names = genes_used  # ✅ 长度必须等于 XT/SV 的列数
assert XT.shape[1] == SV.shape[1] == len(feature_names)

genes_to_plot = ["Stmn4"]
plot_shap_dependence_grid_np(XT, SV, feature_names=feature_names, genes=genes_to_plot, ncols=1, figsize=(4,3))
plt.show()

In [ ]:
import shap

shap.dependence_plot(
    "Gpr85",
    SV,     # shap values
    XT,     # expression matrix
    feature_names=feature_names,
    interaction_index="Stmn4"
)

In [ ]:
# # hyperparamters visualization

# # ========= 1) 单次评估封装 =========
# def eval_one_setting(adata, genes, base_params, override_params, random_state=0):
#     mp = dict(base_params)
#     mp.update(override_params)

#     res = fit_lomo_xgb(
#         adata=adata,
#         genes=genes,
#         model_params=mp,
#         random_state=random_state,
#         verbose=False
#     )
#     fold_df = res["fold_metrics"].copy()

#     # 便于后面做更细分析：保存每折AUC
#     row = {
#         **mp,
#         **{f"fold_auc_{g}": a for g, a in zip(fold_df["group"], fold_df["auc"])},
#         "simple_mean_auc": res["simple_mean_auc"],
#         "weighted_mean_auc": res["weighted_mean_auc"],
#     }
#     return row


# # ========= 2) 单参数敏感性（one-factor-at-a-time） =========
# def sweep_one_param(adata, genes, base_params, param_name, values, random_state=0):
#     rows = []
#     for v in values:
#         rows.append(
#             eval_one_setting(
#                 adata, genes,
#                 base_params=base_params,
#                 override_params={param_name: v},
#                 random_state=random_state
#             )
#         )
#     df = pd.DataFrame(rows)
#     df["param"] = param_name
#     df["value"] = values
#     return df


# # ========= 3) 多参数批量敏感性 =========
# def run_sensitivity(adata, genes, base_params, param_grid, random_state=0):
#     dfs = []
#     for p, vals in param_grid.items():
#         dfs.append(sweep_one_param(adata, genes, base_params, p, vals, random_state=random_state))
#     df_all = pd.concat(dfs, ignore_index=True)
#     return df_all


# # ========= 4) 画图 =========
# def plot_sensitivity(df_all, metric="simple_mean_auc"):
#     # 每个参数单独一张图（最清晰）
#     for p in df_all["param"].unique():
#         d = df_all[df_all["param"] == p].copy()
#         d = d.sort_values("value")

#         plt.figure()
#         plt.plot(d["value"], d[metric], marker="o")
#         plt.xlabel(p)
#         plt.ylabel(f"LOMO {metric}")
#         plt.title(f"Sensitivity: {p} ({metric})")
#         plt.show()


# # ========= 5) 你需要设置：base_params + param_grid =========
# base_params = dict(
#     n_estimators=1000,
#     learning_rate=0.01,
#     max_depth=5,
#     subsample=0.7,
#     colsample_bytree=0.7,
#     reg_lambda=1,
#     eval_metric="logloss",
#     n_jobs=-1,
#     random_state=0
# )

# param_grid = {
#     # 你已经看过的
#     "max_depth": [2, 3, 4, 5, 6, 8, 10],
#     "reg_lambda": [0, 0.5, 1, 2, 5, 10],

#     # 强烈建议补的：泛化/稳定性最敏感的旋钮之一
#     "min_child_weight": [1, 2, 5, 10, 20],

#     # 学习率（如果你没做early stopping，意义略弱；做了early stopping会更关键）
#     "learning_rate": [0.005, 0.01, 0.02, 0.05],

#     # 随机采样（正则化）
#     "subsample": [0.6, 0.7, 0.8, 0.9],
#     "colsample_bytree": [0.6, 0.7, 0.8, 0.9],
# }

# # ========= 6) 运行 =========
# # 把 adata_pn / genes 换成你实际变量名
# df_all = run_sensitivity(adata_pn, genes, base_params, param_grid, random_state=0)

# # 查看每个参数最好的几组
# summary = (
#     df_all[["param", "value", "simple_mean_auc", "weighted_mean_auc"]]
#     .sort_values(["param", "simple_mean_auc"], ascending=[True, False])
# )
# print(summary.groupby("param").head(5))

# # 画图（主看 simple_mean_auc）
# plot_sensitivity(df_all, metric="simple_mean_auc")

# # 如需也看 weighted mean
# plot_sensitivity(df_all, metric="weighted_mean_auc")